# Step 1 — Preprocessing Teks

**Tujuan praktikum**
1. Mahasiswa memahami alur preprocessing teks: *case folding*, tokenisasi, penghapusan *stopword*, dan *stemming*.
2. Mahasiswa mampu mengimplementasikan preprocessing untuk teks berbahasa Indonesia dengan pustaka **Sastrawi**.
3. Mahasiswa mampu mengukur pengaruh tiap tahap preprocessing terhadap jumlah token dan ukuran kosakata.

Korpus praktikum berada di `../data/korpus.csv` dan berisi 30 dokumen berita pendek berbahasa Indonesia dari lima topik: kesehatan, pertanian, teknologi informasi, ekonomi, dan pendidikan.

In [20]:
# Jalankan sekali jika pustaka belum terpasang:
# !pip install pandas numpy matplotlib Sastrawi
import pandas as pd
pd.set_option("display.max_colwidth", 120)

## Langkah 1 — Memuat korpus

In [21]:
korpus = pd.read_csv("../data/korpus.csv")
print("Jumlah dokumen:", len(korpus))
korpus.head()

Jumlah dokumen: 30


,doc_id,judul,isi
0,D01,Demam Berdarah Dengue,Demam berdarah dengue disebabkan oleh virus dengue yang ditularkan melalui gigitan nyamuk Aedes aegypti. Gejala utam...
1,D02,Pencegahan Stunting pada Balita,Stunting adalah kondisi gagal tumbuh pada anak balita akibat kekurangan gizi kronis. Pemenuhan gizi pada seribu hari...
2,D03,Manfaat Olahraga bagi Jantung,"Olahraga teratur seperti jalan cepat, bersepeda, dan berenang dapat menurunkan risiko penyakit jantung. Aktivitas fi..."
3,D04,Mengenal Diabetes Melitus,Diabetes melitus tipe 2 berkaitan erat dengan pola makan tinggi gula dan kurangnya aktivitas fisik. Penderita diabet...
4,D05,Pentingnya Imunisasi Dasar,"Imunisasi dasar lengkap melindungi bayi dari penyakit campak, polio, dan difteri. Vaksin bekerja dengan melatih sist..."


In [22]:
# Judul dan isi digabung menjadi satu teks dokumen
korpus["teks"] = korpus["judul"] + ". " + korpus["isi"]
contoh = korpus.loc[16, "teks"]
print(contoh)

Pemrosesan Bahasa Alami Bahasa Indonesia. Pemrosesan bahasa alami untuk bahasa Indonesia menghadapi tantangan imbuhan yang kompleks. Stemming digunakan untuk mengubah kata berimbuhan menjadi kata dasar, misalnya kata pembelajaran menjadi ajar. Tokenisasi dan penghapusan stopword merupakan tahap awal pengolahan teks.


## Langkah 2 — Case folding
*Case folding* menyeragamkan huruf menjadi huruf kecil sehingga `Stemming`, `stemming`, dan `STEMMING` dianggap kata yang sama.

In [23]:
def case_folding(teks):
    return teks.lower()

print(case_folding(contoh))

pemrosesan bahasa alami bahasa indonesia. pemrosesan bahasa alami untuk bahasa indonesia menghadapi tantangan imbuhan yang kompleks. stemming digunakan untuk mengubah kata berimbuhan menjadi kata dasar, misalnya kata pembelajaran menjadi ajar. tokenisasi dan penghapusan stopword merupakan tahap awal pengolahan teks.


## Langkah 3 — Tokenisasi
Tokenisasi memecah teks menjadi satuan kata (token). Di sini digunakan *regular expression* `[a-z0-9]+`: setiap rangkaian huruf/angka dianggap satu token, sedangkan tanda baca menjadi pemisah.

In [24]:
import re

def tokenize(teks):
    return re.findall(r"[a-z0-9]+", teks.lower())

token_contoh = tokenize(case_folding(contoh))
print(len(token_contoh), "token")
print(token_contoh)

39 token
['pemrosesan', 'bahasa', 'alami', 'bahasa', 'indonesia', 'pemrosesan', 'bahasa', 'alami', 'untuk', 'bahasa', 'indonesia', 'menghadapi', 'tantangan', 'imbuhan', 'yang', 'kompleks', 'stemming', 'digunakan', 'untuk', 'mengubah', 'kata', 'berimbuhan', 'menjadi', 'kata', 'dasar', 'misalnya', 'kata', 'pembelajaran', 'menjadi', 'ajar', 'tokenisasi', 'dan', 'penghapusan', 'stopword', 'merupakan', 'tahap', 'awal', 'pengolahan', 'teks']


In [25]:
# Bandingkan dengan tokenisasi naif menggunakan split() — tanda baca masih menempel
print(case_folding(contoh).split()[:12])

['pemrosesan', 'bahasa', 'alami', 'bahasa', 'indonesia.', 'pemrosesan', 'bahasa', 'alami', 'untuk', 'bahasa', 'indonesia', 'menghadapi']


## Langkah 4 — Penghapusan stopword
*Stopword* adalah kata yang sangat sering muncul tetapi hampir tidak membawa makna topik, misalnya *yang, dan, di, untuk*. Sastrawi menyediakan daftar stopword bahasa Indonesia.

In [28]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

STOPWORDS = set(StopWordRemoverFactory().get_stop_words())
print("Jumlah stopword:", len(STOPWORDS))
print(sorted(STOPWORDS)[:40])

Jumlah stopword: 123
['ada', 'adalah', 'agak', 'agar', 'akan', 'amat', 'anda', 'antara', 'anu', 'apakah', 'apalagi', 'atau', 'bagaimanapun', 'bagi', 'bahwa', 'begitu', 'belum', 'bisa', 'boleh', 'dahulu', 'dalam', 'dan', 'dapat', 'dari', 'daripada', 'demi', 'demikian', 'dengan', 'di', 'dia', 'dimana', 'dll', 'dsb', 'dst', 'dua', 'dulunya', 'guna', 'hal', 'hanya', 'harus']


In [29]:
def remove_stopwords(tokens, stopwords=STOPWORDS):
    return [t for t in tokens if t not in stopwords]

tanpa_sw = remove_stopwords(token_contoh)
print(len(token_contoh), "->", len(tanpa_sw), "token")
print(tanpa_sw)

39 -> 35 token
['pemrosesan', 'bahasa', 'alami', 'bahasa', 'indonesia', 'pemrosesan', 'bahasa', 'alami', 'bahasa', 'indonesia', 'menghadapi', 'tantangan', 'imbuhan', 'kompleks', 'stemming', 'digunakan', 'mengubah', 'kata', 'berimbuhan', 'menjadi', 'kata', 'dasar', 'misalnya', 'kata', 'pembelajaran', 'menjadi', 'ajar', 'tokenisasi', 'penghapusan', 'stopword', 'merupakan', 'tahap', 'awal', 'pengolahan', 'teks']


## Langkah 5 — Stemming
*Stemming* mengubah kata berimbuhan menjadi kata dasar. Bahasa Indonesia kaya imbuhan (awalan, akhiran, sisipan, konfiks), sehingga Sastrawi memakai algoritma berbasis kamus kata dasar (Nazief–Adriani yang diperluas).

In [30]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

stemmer = StemmerFactory().create_stemmer()
for kata in ["pembelajaran", "pertanian", "petani", "pencarian", "keamanan",
             "memantau", "berolahraga", "perekonomian", "mahasiswa"]:
    print(f"{kata:15s} -> {stemmer.stem(kata)}")

pembelajaran    -> ajar
pertanian       -> tani
petani          -> tani
pencarian       -> cari
keamanan        -> aman
memantau        -> pantau
berolahraga     -> olahraga
perekonomian    -> ekonomi
mahasiswa       -> mahasiswa


Stemming Sastrawi relatif lambat karena setiap kata dicocokkan ke kamus. Karena kata yang sama muncul berulang kali, hasilnya disimpan dalam *cache* (dictionary) agar setiap kata unik hanya di-stem sekali.

In [31]:
_cache = {}

def stem(token):
    if token not in _cache:
        _cache[token] = stemmer.stem(token)
    return _cache[token]

print([stem(t) for t in tanpa_sw])

['pemrosesan', 'bahasa', 'alami', 'bahasa', 'indonesia', 'pemrosesan', 'bahasa', 'alami', 'bahasa', 'indonesia', 'hadap', 'tantang', 'imbuh', 'kompleks', 'stemming', 'guna', 'ubah', 'kata', 'imbuh', 'jadi', 'kata', 'dasar', 'misal', 'kata', 'ajar', 'jadi', 'ajar', 'tokenisasi', 'hapus', 'stopword', 'rupa', 'tahap', 'awal', 'olah', 'teks']


## Langkah 6 — Menggabungkan menjadi satu fungsi `preprocess`

In [32]:
def preprocess(teks, use_stopword=True, use_stemming=True):
    tokens = tokenize(case_folding(teks))
    if use_stopword:
        tokens = remove_stopwords(tokens)
    if use_stemming:
        tokens = [stem(t) for t in tokens]
    return tokens

korpus["tokens"] = korpus["teks"].apply(preprocess)
korpus[["doc_id", "judul", "tokens"]].head()

,doc_id,judul,tokens
0,D01,Demam Berdarah Dengue,"[demam, darah, dengue, demam, darah, dengue, sebab, virus, dengue, tular, lalu, gigit, nyamuk, aedes, aegypti, gejal..."
1,D02,Pencegahan Stunting pada Balita,"[cegah, stunting, balita, stunting, kondisi, gagal, tumbuh, anak, balita, akibat, kurang, gizi, kronis, penuh, gizi,..."
2,D03,Manfaat Olahraga bagi Jantung,"[manfaat, olahraga, jantung, olahraga, atur, jalan, cepat, sepeda, renang, turun, risiko, sakit, jantung, aktivitas,..."
3,D04,Mengenal Diabetes Melitus,"[kenal, diabetes, melitus, diabetes, melitus, tipe, 2, kait, erat, pola, makan, tinggi, gula, kurang, aktivitas, fis..."
4,D05,Pentingnya Imunisasi Dasar,"[penting, imunisasi, dasar, imunisasi, dasar, lengkap, lindung, bayi, sakit, campak, polio, difteri, vaksin, kerja, ..."


## Langkah 7 — Mengukur pengaruh tiap tahap

In [33]:
tahap = {
    "Tokenisasi saja": dict(use_stopword=False, use_stemming=False),
    "+ Stopword removal": dict(use_stopword=True, use_stemming=False),
    "+ Stemming (tanpa stopword removal)": dict(use_stopword=False, use_stemming=True),
    "+ Stopword removal + Stemming": dict(use_stopword=True, use_stemming=True),
}
baris = []
for nama, opsi in tahap.items():
    semua = [t for teks in korpus["teks"] for t in preprocess(teks, **opsi)]
    baris.append({"Tahap": nama, "Jumlah token": len(semua), "Ukuran kosakata": len(set(semua))})
pd.DataFrame(baris)

,Tahap,Jumlah token,Ukuran kosakata
0,Tokenisasi saja,1033,574
1,+ Stopword removal,863,541
2,+ Stemming (tanpa stopword removal),1033,501
3,+ Stopword removal + Stemming,863,472


In [34]:
from collections import Counter

frekuensi = Counter(t for tokens in korpus["tokens"] for t in tokens)
pd.DataFrame(frekuensi.most_common(15), columns=["term", "frekuensi"])

,term,frekuensi
0,tani,12
1,guna,12
2,ajar,11
3,digital,11
4,lalu,9
5,data,9
6,bantu,8
7,tinggi,7
8,jadi,7
9,kata,7


## Langkah 8 — Menyimpan fungsi untuk step berikutnya
Fungsi yang sama sudah disediakan di `stki_utils.py` agar notebook step berikutnya cukup mengimpornya. Sel berikut memastikan hasil fungsi Anda identik dengan versi di `stki_utils.py`.

In [35]:
import stki_utils

cocok = all(preprocess(t) == stki_utils.preprocess(t) for t in korpus["teks"])
print("Hasil identik dengan stki_utils.preprocess:", cocok)

Hasil identik dengan stki_utils.preprocess: True


## Tugas Step 1
1. Temukan minimal **tiga** kasus *overstemming* atau *understemming* Sastrawi pada korpus (misalnya dua kata berbeda makna yang menghasilkan stem sama). Jelaskan dampaknya terhadap pencarian.
2. Tambahkan stopword khusus domain (misalnya `dapat`, `membantu`) ke daftar stopword. Bandingkan ukuran kosakata sebelum dan sesudahnya.
3. Ubah tokenizer agar kata berulang seperti `anak-anak` atau `sayur-mayur` tetap menjadi satu token. Uji pada kalimat buatan Anda sendiri.
4. Tuliskan analisis singkat (maksimal 1 paragraf): tahap mana yang paling banyak mengurangi ukuran kosakata, dan mengapa?

In [15]:
# Tulis jawaban Anda di sini